# Confidence Analysis per Joint

## Models: `MoveNet` and `MediaPipe`

## **Calculate Mean and Std for each joint**

In [61]:
import pandas as pd
import numpy as np
import glob
import os

In [62]:
BASE_DIR = '../data/processed/keypoints_angles'
OUT_DIR  = '../data/processed/confidence_stats'
os.makedirs(OUT_DIR, exist_ok=True)

JOINTS = [
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
]

In [63]:
movenet_files = sorted(glob.glob(f'{BASE_DIR}/movenet/*_movenet_angles.csv'))
mediapipe_files = sorted(glob.glob(f'{BASE_DIR}/mediapipe_norm/*_mediapipe_norm_angles.csv'))

df_movenet = pd.concat([pd.read_csv(f) for f in movenet_files], ignore_index=True)
df_mediapipe = pd.concat([pd.read_csv(f) for f in mediapipe_files], ignore_index=True)

print(f"MoveNet Data Loaded:    {len(df_movenet)} total frames")
print(f"MediaPipe Data Loaded:  {len(df_mediapipe)} total frames")

MoveNet Data Loaded:    15010 total frames
MediaPipe Data Loaded:  15010 total frames


## MoveNet
- **Confidence Analysis per joint**
- **Uses confidence score**

In [64]:
rows = []
for j in JOINTS:
    rows.append({
        'Joint': j.replace('_', ' ').title(),
        'Mean': df_movenet[f'{j}_confidence'].mean(),
        'Std':  df_movenet[f'{j}_confidence'].std()
    })

table_mn = pd.DataFrame(rows).round(4)
table_mn.to_csv(f'{OUT_DIR}/movenet_per_joint.csv', index=False)

print("MoveNet Confidence:")
display(table_mn)

MoveNet Confidence:


,Joint,Mean,Std
0,Left Shoulder,0.7900,0.0890
1,Right Shoulder,0.7868,0.0857
2,Left Elbow,0.7185,0.1462
3,Right Elbow,0.7003,0.1380
4,Left Wrist,0.5780,0.1579
5,Right Wrist,0.5700,0.1589
6,Left Hip,0.8017,0.0922
7,Right Hip,0.7419,0.0966
8,Left Knee,0.6912,0.1530
9,Right Knee,0.7927,0.0994


## MediaPipe 
- **Visibility & presence per joint**
- **Uses visibility and presence scores**

In [65]:
rows = []
for j in JOINTS:
    rows.append({
        'Joint': j.replace('_', ' ').title(),
        'Vis Mean': df_mediapipe[f'{j}_visibility'].mean(),
        'Vis Std':  df_mediapipe[f'{j}_visibility'].std(),
        'Pres Mean': df_mediapipe[f'{j}_presence'].mean(),
        'Pres Std':  df_mediapipe[f'{j}_presence'].std()
    })

table_mp = pd.DataFrame(rows).round(4)
table_mp.to_csv(f'{OUT_DIR}/mediapipe_per_joint.csv', index=False)

print("MediaPipe Visibility & Presence:")
display(table_mp)

MediaPipe Visibility & Presence:


,Joint,Vis Mean,Vis Std,Pres Mean,Pres Std
0,Left Shoulder,1.0000,0.0000,1.0000,0.0000
1,Right Shoulder,0.9999,0.0001,1.0000,0.0000
2,Left Elbow,0.9454,0.1087,0.9999,0.0002
3,Right Elbow,0.5205,0.1973,0.9999,0.0002
4,Left Wrist,0.9080,0.1721,1.0000,0.0000
5,Right Wrist,0.6510,0.2729,1.0000,0.0000
6,Left Hip,1.0000,0.0000,1.0000,0.0000
7,Right Hip,1.0000,0.0000,1.0000,0.0000
8,Left Knee,0.9601,0.0844,1.0000,0.0000
9,Right Knee,0.7843,0.1911,1.0000,0.0000


## Overall Stats

In [66]:
mn_cols   = [f'{j}_confidence' for j in JOINTS]
vis_cols  = [f'{j}_visibility' for j in JOINTS]
pres_cols = [f'{j}_presence'   for j in JOINTS]

all_mn_vals      = df_movenet[mn_cols].values.flatten()
all_mp_vis_vals  = df_mediapipe[vis_cols].values.flatten()
all_mp_pres_vals = df_mediapipe[pres_cols].values.flatten()

In [67]:
# Summary Table
summary = pd.DataFrame({
    'Model Metric': ['MoveNet (Confidence)', 'MediaPipe (Visibility)', 'MediaPipe (Presence)'],
    'Mean': [np.nanmean(all_mn_vals), np.nanmean(all_mp_vis_vals), np.nanmean(all_mp_pres_vals)],
    'Std':  [np.nanstd(all_mn_vals),  np.nanstd(all_mp_vis_vals),  np.nanstd(all_mp_pres_vals)]
}).round(4)

summary.to_csv(f'{OUT_DIR}/overall_summary.csv', index=False)

print("Overall Summary (All Data Flattened):")
display(summary)

Overall Summary (All Data Flattened):


,Model Metric,Mean,Std
0,MoveNet (Confidence),0.7321,0.1452
1,MediaPipe (Visibility),0.8823,0.2031
2,MediaPipe (Presence),1.0000,0.0001
